# 5-11 IR정보 가져오기 — 회사가 올린 IR 자료와 PDF 본문

**질문 예시:** "이번 주 IR 자료 올린 회사 목록 줘", "삼성전자 최근 1년 IR 자료랑 최신 실적발표 PDF 내용 보여줘"

DeepSearch `["company"],["ir"]`에는 상장사가 올린 **IR 자료(실적발표·회사소개 PDF)**가 들어 있습니다.
문서마다 회사(`securities`)와 **PDF 링크**(`content_url`)가 붙어 있어, PDF를 받아 앞 장 텍스트를 바로 읽을 수 있습니다.
(IR 개최 '안내 공시'는 공시 쪽에 있습니다 → 5-10)

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/5_data_lookup/11_ir.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
import numpy as np
ASOF = "2026-11-24"
STOCK = "005930"

if IN_COLAB:
    !pip -q install pymupdf

def doc_pages(category, section, query, date_from, date_to, pages=1, count=100):
    """doc_search 와 같은 표를 주되, 페이지가 count보다 적게 와도 전체 건수에 닿을 때까지 다음 페이지를 받습니다.
    (서버가 중복을 빼고 주는지 한 페이지가 80~90건으로 오는 일이 흔합니다.)"""
    raw, total = [], 0
    for p in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},{json.dumps(query, ensure_ascii=False)},'
             f'count={count},page={p},date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        raw += docs
        if not docs or p * count >= total:
            break
    rows = [{"created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
             "publisher": x.get("publisher"), "title": " ".join(str(x.get("title") or "").split()),
             "stocks": ", ".join(s.get("name", "") for s in x.get("securities") or []),
             "symbols": ", ".join(s.get("symbol", "") for s in x.get("securities") or []),
             "polarity": _polarity(x), "url": x.get("content_url")} for x in raw]
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols).drop_duplicates("url")

def ir_docs(query, date_from, date_to, pages=1):
    total, df = doc_pages("company", ["ir"], query, date_from, date_to, pages=pages)
    print(f"IR '{query}' {date_from}~{date_to}: 전체 {total:,}건 중 {len(df)}건")
    return df

## 1. 최근 2주 올라온 IR 자료 전체

In [4]:
d2w = (pd.Timestamp(ASOF) - pd.Timedelta(days=13)).date()
recent = ir_docs("*", d2w, ASOF, pages=4)
u = universe(); u["code"] = u.symbol.str.replace("KRX:", "", regex=False)
recent["code"] = recent.symbols.str.split(", ").str[0]
recent = recent.merge(u[["code", "market"]], on="code", how="left")
recent["자료 성격"] = np.select(
    [recent.title.str.contains("실적|결산|분기|Earnings", case=False, regex=True),
     recent.title.str.contains("회사소개|IR|사업|Company|Corporate", case=False, regex=True)],
    ["실적", "회사소개·사업"], "기타")
print("시장별:", recent.market.fillna("기타").value_counts().to_dict())
print("자료 성격(제목 기준):", recent["자료 성격"].value_counts().to_dict())
recent[["created_at", "stocks", "market", "title"]].head(10)

IR '*' 2026-11-11~2026-11-24: 전체 317건 중 317건


시장별: {'KOSPI': 163, 'KOSDAQ': 150, '기타': 4}
자료 성격(제목 기준): {'기타': 138, '실적': 111, '회사소개·사업': 68}


,created_at,stocks,market,title
0,2026-11-24 15:00:00,에스지에이솔루션즈,KOSDAQ,회사소개 및 사업현황 업데이트
1,2026-11-24 14:43:19,이랜텍,KOSDAQ,2026년 3분기 실적
2,2026-11-24 14:41:04,네패스,KOSDAQ,교보증권 NDR
3,2026-11-24 14:39:27,레드캡투어,KOSDAQ,회사소개 및 사업현황 설명
4,2026-11-24 14:37:12,브리지텍,KOSDAQ,회사현황 소개
5,2026-11-24 14:35:40,쿠콘,KOSDAQ,2026년 3분기 경영실적
6,2026-11-24 14:33:23,에어부산,KOSPI,합병 관련 주요 내용 설명
7,2026-11-24 14:25:55,메디포스트,KOSDAQ,미래에셋증권 헬스케어 Corporate Day
8,2026-11-24 14:12:09,코미코,KOSDAQ,회사소개 및 경영현황 설명
9,2026-11-24 13:53:57,티씨케이,KOSDAQ,Goldman Sachs - KRX KOSDAQ Corporate Day 2026


## 2. 종목별 — 최근 1년 IR 자료

In [5]:
d1y = (pd.Timestamp(ASOF) - pd.DateOffset(years=1) + pd.Timedelta(days=1)).date()
one = ir_docs(f"securities.symbol:{STOCK}", d1y, ASOF)
one[["created_at", "title", "url"]]

IR 'securities.symbol:005930' 2025-11-25~2026-11-24: 전체 5건 중 5건


,created_at,title,url
0,2026-10-29 13:38:39,2026년 3분기 경영실적 발표,https://ddi-cdn.deepsearch.com/company/ir/2026/10/29/176...
1,2026-09-08 10:25:11,모듈러 주택 생산에 휴머노이드 투입…‘건설 자동화’ 불 지핀다,https://ddi-cdn.deepsearch.com/company/ir/2026/09/08/174...
2,2026-07-30 17:41:59,2026년 2분기 경영실적 발표,https://ddi-cdn.deepsearch.com/company/ir/2026/07/30/173...
3,2026-04-30 13:15:33,2026년 1분기 경영실적 발표,https://ddi-cdn.deepsearch.com/company/ir/2026/04/30/170...
4,2026-01-29 13:49:46,2025년 4분기 경영실적 발표,https://ddi-cdn.deepsearch.com/company/ir/2026/01/29/166...


## 3. IR 자료를 많이 올린 회사 (최근 3개월)
건수만 필요하면 `DocumentAggregation` 한 번이면 됩니다.

In [6]:
d3m = (pd.Timestamp(ASOF) - pd.DateOffset(months=3)).date()
c = ds(f'DocumentAggregation(["company"],["ir"],"*","securities.symbol:200",date_from={d3m},date_to={ASOF})')
agg = pd.DataFrame(c["data"]["pods"][1]["content"]["data"])[["key", "count"]]
agg = agg.merge(u[["code", "name", "market"]], left_on="key", right_on="code", how="inner")
print(f"3개월간 IR 자료를 올린 상장사 {len(agg)}곳, 자료 {agg['count'].sum():,}건")
agg[["name", "market", "count"]].rename(columns={"count": "IR 자료 수"}).head(12)

3개월간 IR 자료를 올린 상장사 196곳, 자료 779건


,name,market,IR 자료 수
0,한국콜마,KOSPI,17
1,현대모비스,KOSPI,15
2,래블업,KOSDAQ,15
3,코스맥스,KOSPI,14
4,대신증권,KOSPI,11
5,미래에셋증권,KOSPI,11
6,신한지주,KOSPI,11
7,아이센스,KOSDAQ,10
8,노을,KOSDAQ,10
9,에이직랜드,KOSDAQ,10


## 4. PDF 앞 장 읽기
가장 최근 IR 자료 PDF를 받아 텍스트를 뽑습니다(PyMuPDF). 실적발표 자료는 1~2쪽이 표지·유의사항이라 3쪽부터 읽습니다.
표·그림 위주 슬라이드는 텍스트가 적게 나옵니다.

In [7]:
import fitz  # PyMuPDF

def pdf_text(url, start=1, pages=2, limit=1500):
    """PDF의 start쪽(1부터)부터 pages쪽 텍스트."""
    r = requests.get(url, timeout=60); r.raise_for_status()
    doc = fitz.open(stream=r.content, filetype="pdf")
    idx = range(start - 1, min(start - 1 + pages, doc.page_count))
    text = "\n".join(doc[i].get_text() for i in idx)
    print(f"{doc.page_count}쪽 PDF, {start}쪽부터 {len(idx)}쪽 {len(text):,}자")
    return " ".join(text.split())[:limit]

latest = one.sort_values("created_at").iloc[-1]
print(latest.created_at, latest.title)
print(pdf_text(latest.url, start=3, pages=2))

2026-10-29 13:38:39 2026년 3분기 경영실적 발표


16쪽 PDF, 3쪽부터 2쪽 311자
※ 배경사진: 화성캠퍼스DSR (Device Solutions Research) 타워, 대한민국 목차 발표자소개 별첨 '26년3분기경영실적 사업부문별 실적및전망 Device eXperience (DX) Device Solutions (DS) 발표자 진행 김재준부사장 메모리 신승철부사장 System LSI 강석채부사장 Foundry 허철부사장 삼성디스플레이 김원우상무 영상디스플레이(VD) 다니엘오부사장 Investor Relations 박순철부사장 Chief Financial Officer 발표자소개 다니엘아라우조상무 Mobile eXperience


## 결과 읽는 법과 한계
- **IR 자료는 회사가 올린 것만** 있습니다. 모든 상장사가 올리지는 않으며, 같은 자료의 국문·영문본이 따로 잡히기도 합니다.
- **기사형 문서가 섞입니다.** 회사 홍보 기사가 IR로 분류된 경우가 있습니다(제목으로 구분).
- PDF 원문은 DeepSearch CDN 링크라 따로 키가 필요 없습니다. 스캔 이미지 PDF는 텍스트가 나오지 않습니다.